In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_tpsa)=
# Get TPSA

*Calculating topological polar surface areas for complete molecular graphs.*

The experimental {func}`molsysmt.physchem.get_tpsa` implements the Ertl,
Rohde and Selzer model through RDKit. It returns one area per selected molecule.
TPSA is a graph descriptor, not geometric SASA or a permeability prediction.
No conformer generation, protonation preparation or source mutation occurs.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_tpsa` for input requirements and diagnostics.
:::


## Basic usage

Let's show how this function works with two fully defined chemical graphs.
RDKit is an optional explicit provider. Inputs may use any supported form with
the required chemistry and molecule membership. No coordinates are needed.

In [2]:
import molsysmt as msm
from rdkit import Chem

molsys = msm.convert(Chem.MolFromSmiles('CCO.COC'), to_form='molsysmt.MolSys')
area = msm.physchem.get_tpsa(molsys)
msm.pyunitwizard.get_value(area, to_unit='angstrom**2').round(2).tolist()

[20.23, 9.23]

## Selecting whole molecules

Numeric selections are molecule indices and retain order and repetitions.
String expressions use `syntax`; an expression identifying an atom selects its
whole parent molecule rather than cutting a chemically different fragment.
The only supported `element` is `molecule`. Several molecules yield separate
values, with no automatic system-level sum.

In [3]:
repeated = msm.physchem.get_tpsa(molsys, selection=[1, 0, 1], return_report=True)
selected = msm.physchem.get_tpsa(molsys, selection='atom_index==0')
repeated['report']['molecule_indices'].tolist(), repeated['report']['atom_offsets'].tolist(), selected.shape

([1, 0, 1], [0, 3, 6, 9], (1,))

## Choosing the polar elements

The default counts N/O contributions, following the conventional reference
program/RDKit variant. Set `include_sulfur_phosphorus=True` to include polar
S/P contributions from the published fragment table. Keep this choice explicit
when comparing values or building a dataset. Neither variant is universally better.

In [4]:
molsys = msm.convert(Chem.MolFromSmiles('OC(=O)P(=O)(O)O'), to_form='molsysmt.MolSys')
{
    enabled: msm.pyunitwizard.get_value(
        msm.physchem.get_tpsa(molsys, include_sulfur_phosphorus=enabled),
        to_unit='angstrom**2',
    ).round(2).tolist()
    for enabled in (False, True)
}

{False: [94.83], True: [104.64]}

## Using the active unit policy

The area is a PyUnitWizard quantity: 20.23 angstrom squared corresponds to
0.2023 nm squared. Unlike bare descriptor numbers, its value and unit remain
coupled when the application changes its standard length unit.

In [5]:
molsys = msm.convert(Chem.MolFromSmiles('CCO'), to_form='molsysmt.Topology')
with msm.pyunitwizard.context(standard_units=['pm', 'ps', 'radians']):
    area = msm.physchem.get_tpsa(molsys)
    empty = msm.physchem.get_tpsa(molsys, selection=[])
    print(msm.pyunitwizard.get_unit(area))
    print(msm.pyunitwizard.get_value(area, to_unit='nm**2').round(4).tolist())
    print(empty.shape)

picometer ** 2
[0.2023]
(0,)


## Declaring the chemical state

`chemical_state='reference'` uses the declared reference, not the state of the
first selected structure. An integer chooses a stored state. A `structure`
request must resolve unambiguously across `structure_indices`. This graph-based
property has no structures output axis. Spatial selection may require coordinates.

Charged molecular graphs are allowed, but charges, bond orders and the hydrogen
inventory must already be defined. Indexed H and declared nonindexed H are
supported; the function never prepares a missing protonation state. Native SDF
inputs need explicit valence/H evidence. A default CTAB valence does not supply
that evidence; such inputs fail clearly rather than receiving inferred counts.

In [6]:
molsys = msm.convert(Chem.MolFromSmiles('CC(=O)[O-]'), to_form='molsysmt.MolSys')
result = msm.physchem.get_tpsa(molsys, chemical_state='reference', return_report=True)
result['report']['formal_charges'].tolist(), msm.pyunitwizard.get_value(result['tpsa'], to_unit='angstrom**2').round(2).tolist()

([-1], [40.13])

## Retaining the provenance

`return_report=True` returns `tpsa` and a detached `report`. Source molecule
indices and flattened atom membership with offsets identify the evaluated
population. The report records the state/digest, method, element scope,
hydrogen policy, original software versions and bibliography. The quantity
retains its unit; the metadata report alone does not contain area values.
No automatic H5MSM attachment or standalone JSON serialization is implied.
Optional Ackredit contributes to the application's session and preserves
completed results if attribution is unavailable or fails.

In [7]:
report = result['report']
print(report['method'], report['parameters'])
print(report['atom_indices'].tolist(), report['atom_offsets'].tolist())
print([item['id'] for item in report['attribution']['items']])

ertl_rohde_selzer {'include_sulfur_phosphorus': False, 'hydrogen_treatment': 'preserve_declared_inventory', 'implementation': 'rdkit.CalcTPSA'}
[0, 1, 2, 3] [0, 4]
['rdkit-aromaticity-model', 'software:molsysmt:0.23.0+201.g3502232d8.dirty', 'software:rdkit:2025.09.5', 'doi:10.1021/jm000942e']


## Understanding the limits

A selected molecule must be covalently complete and connected. A label containing
several salt components, or a covalent bond crossing a molecule boundary, fails.
The initial route excludes metals, dative bonds within a molecule, radicals and
query/dummy atoms. Chemical contradictions and missing evidence fail explicitly.
Geometry, solvent shielding and environmental pKa are outside this model.

The method reference is [Ertl, Rohde and Selzer (2000)](https://doi.org/10.1021/jm000942e).
The [RDKit Book](https://www.rdkit.org/docs/RDKit_Book.html#implementation-of-the-tpsa-descriptor)
explains the two polar-element variants. Original-model attribution is distinct
from the executed RDKit implementation.

:::{seealso}
:class: dropdown

- {func}`molsysmt.basic.convert`: supplying an appropriate molecular form.
- {func}`molsysmt.physchem.get_tpsa`: calculating the graph descriptor.
- {func}`molsysmt.physchem.get_sasa`: calculating geometric solvent exposure.
- {ref}`Cookbook_TPSA_Descriptors`: exporting indexed molecular descriptors.
:::
